In [ ]:
import torch
import os
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor,CLIPTextModel, get_linear_schedule_with_warmup

In [ ]:
TEACHER_ID = "openai/clip-vit-base-patch32" # Output dim: 512
STUDENT_ID = "IRIISNEPAL/RoBERTa_Nepali_110M" # Output dim: 768
# STUDENT_ID = "Shushant/nepaliBERT" # Output dim: 768
MAX_LENGTH = 77  # CLIP's hard limit
BATCH_SIZE = 64
EPOCHS = 4
LEARNING_RATE = 5e-5
WARMUP_STEPS = 1000
SAVE_EVERY = 10000  # Save a checkpoint every 10000 steps
VAL_EVERY = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_SAVE_PATH = "/content/drive/MyDrive/CC3M_MSCOCO_Distillation_Preprocessing/"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

#Loading of Training and Validation Dataset

In [ ]:
!mkdir -p /content/dataset/train
!cp -r "/content/drive/MyDrive/TrainingData/." "/content/dataset/train"

In [ ]:
!mkdir -p /content/dataset/val
!cp -r "/content/drive/MyDrive/CLIP_Results_Val_IndicTrans2/." "/content/dataset/val"

In [ ]:
!cp -r "/content/drive/MyDrive/MSCOCO_Results_Val_IndicTrans2/." "/content/dataset/val"

In [ ]:
import pandas as pd
from datasets import Dataset
import glob
from tqdm.auto import tqdm # Progress bar

def load_dataset(path):
  # 1. Get all parquet files
  all_files = glob.glob(f"{path}/*.parquet")

  # 2. Read with a progress bar
  print(f"Reading {len(all_files)} parquet shards...")
  df_list = []
  for f in tqdm(all_files, desc="Loading Parquets"):
      df_list.append(pd.read_parquet(f))

  # Combine all shards into one DataFrame
  df = pd.concat(df_list)

  # 3. Select and Rename
  df = df[["text", "translated_text"]]
  df = df.rename(columns={"text": "en", "translated_text": "ne"})

  # 4. Convert to Hugging Face Dataset
  # We use Dataset.from_dict or from_pandas.
  # from_pandas is usually fast, but for very large DFs, you can see progress here too:
  print("Converting to Hugging Face Dataset...")
  dataset = Dataset.from_pandas(df, preserve_index=False)

  print("\n--- Success! ---")
  print("First example:", dataset[0])
  print("Total rows:", len(dataset))
  return dataset

In [ ]:
train_dataset = load_dataset("/content/dataset/train")


In [ ]:
val_dataset = load_dataset("/content/dataset/val")

In [ ]:
class MultilingualDistiller(nn.Module):
  def __init__(self, student_id, teacher_id):
      super().__init__()
      self.teacher = CLIPModel.from_pretrained(teacher_id)
      self.teacher.eval()
      for param in self.teacher.parameters():
          param.requires_grad = False
      self.student = AutoModel.from_pretrained(student_id)

      student_hidden_size = self.student.config.hidden_size
      teacher_hidden_size = self.teacher.config.projection_dim
      print(f"Projection Layer: Student {student_hidden_size} -> Teacher {teacher_hidden_size}")
      # This handles the size mismatch
      self.projection = nn.Linear(student_hidden_size, teacher_hidden_size)

  def get_teacher_text_embeddings(self, teacher_inputs):
    with torch.no_grad():
      teacher_model_output = self.teacher.get_text_features(**teacher_inputs)
    return teacher_model_output.pooler_output


  def get_student_text_embeddings(self, student_inputs):
    text_embeddings = self.student(**student_inputs)
    return text_embeddings.pooler_output

  def forward(self, teacher_inputs,student_inputs):
    with torch.no_grad():
      teacher_embeddings = self.get_teacher_text_embeddings(teacher_inputs=teacher_inputs)
    student_embeddings = self.get_student_text_embeddings(student_inputs=student_inputs)
    student_projected_embeddings = self.projection(student_embeddings)

    return teacher_embeddings, student_projected_embeddings

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Frozen Parameters:    {total_params - trainable_params:,}")

In [ ]:
# 4. Initialization
teacher_processor = CLIPProcessor.from_pretrained(TEACHER_ID)
s_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)

model = MultilingualDistiller(STUDENT_ID, TEACHER_ID).to(device)
mse_loss = nn.MSELoss()
count_parameters(model)

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)

def lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda)

In [ ]:
def tokenize_fn(batch):
  teacher_encodings = teacher_processor(
        text=batch["en"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

  student_encodings = s_tokenizer(
        text=batch["ne"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

  return {
      "teacher_input_ids": teacher_encodings["input_ids"],
      "teacher_attention_mask": teacher_encodings["attention_mask"],
      "student_input_ids": student_encodings["input_ids"],
      "student_attention_mask": student_encodings["attention_mask"]
  }

In [ ]:
train_processed_dataset = train_dataset.map(
    tokenize_fn,
    batched=True,
    desc="Tokenizing dataset"
)

val_processed_dataset = val_dataset.map(
    tokenize_fn,
    batched=True,
    desc="Tokenizing dataset"
)

train_processed_dataset.set_format(type="torch", columns=["teacher_input_ids", "teacher_attention_mask",
                                                   "student_input_ids", "student_attention_mask"])

val_processed_dataset.set_format(type="torch", columns=["teacher_input_ids", "teacher_attention_mask",
                                                   "student_input_ids", "student_attention_mask"])

In [ ]:
num_workers = min(4, os.cpu_count() or 1)
train_loader = DataLoader(
    train_processed_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_processed_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)

In [ ]:
def run_validation(model, loader, device, mse_loss):
    """Shows a progress bar, but prints no text/metrics to console."""
    model.eval()
    total_loss = 0
    total_cos = 0

    # leave=False ensures the bar disappears completely when finished
    # removed ncols or extra padding to keep it minimal
    val_pbar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for batch in val_pbar:
            t_tok = {'input_ids': batch['teacher_input_ids'].to(device),
                     'attention_mask': batch['teacher_attention_mask'].to(device)}
            s_tok = {'input_ids': batch['student_input_ids'].to(device),
                     'attention_mask': batch['student_attention_mask'].to(device)}

            t_embs, s_embs = model(t_tok, s_tok)

            loss = mse_loss(s_embs, t_embs)
            cos = F.cosine_similarity(s_embs, t_embs, dim=-1).mean().item()

            total_loss += loss.item()
            total_cos += cos

            val_pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
               "cosine_sim": f"{cos:.4f}",
            })

    avg_loss = total_loss / len(loader)
    avg_cos = total_cos / len(loader)

    val_pbar.set_postfix({
                "avg_loss": f"{avg_loss:.4f}",
               "avg_cos": f"{avg_cos:.4f}",
            })

    val_pbar.close()
    model.train()
    return avg_loss, avg_cos

In [ ]:
from tqdm.notebook import tqdm
import os
from itertools import islice
history = []
global_step = 0
steps_per_epoch = len(train_loader)
for epoch in range(EPOCHS):
    model.train()
    train_pbar = tqdm(train_loader,total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS} - Training")

    for batch in train_pbar:
        global_step += 1
        optimizer.zero_grad()

        t_tok = {
            'input_ids': batch['teacher_input_ids'].to(device),
            'attention_mask': batch['teacher_attention_mask'].to(device)
        }
        s_tok = {
            'input_ids': batch['student_input_ids'].to(device),
            'attention_mask': batch['student_attention_mask'].to(device)
        }

        # Forward pass
        t_embs, s_embs = model(t_tok, s_tok)

        # Minimize MSE between projected student and fixed teacher
        loss = mse_loss(s_embs, t_embs)

        with torch.no_grad():
            train_cos = F.cosine_similarity(s_embs, t_embs, dim=-1).mean().item()

        # Backpropagate through Student and Projection
        loss.backward()
        optimizer.step()
        scheduler.step() # This increments the LR based on global_step

        # Logging
        current_lr = scheduler.get_last_lr()[0]

        # Update progress bar
        train_pbar.set_postfix({
            "step": global_step,
            "loss": f"{loss.item():.4f}",
            "cosine_sim": f"{train_cos:.4f}",
            "lr": f"{current_lr:.2e}"
        })

        step_log = {
            "step": global_step,
            "epoch": epoch,
            "train_loss": loss.item(),
            "train_cos": train_cos,
            "lr": current_lr,
            "avg_train_loss": None,
            "avg_train_cos": None,
            "avg_val_loss": None,
            "avg_val_cos": None
        }

        # 3. Validation Step
        if global_step % VAL_EVERY == 0:
            avg_val_loss, avg_val_cos = run_validation(model, val_loader, device, mse_loss)

            sum_train_loss = sum(log["train_loss"] for log in history) + loss.item()
            sum_train_cos = sum(log["train_cos"] for log in history) + train_cos

            avg_train_loss = sum_train_loss / global_step
            avg_train_cos = sum_train_cos / global_step

            # Update the current log entry with validation data
            step_log["avg_train_loss"] = avg_train_loss
            step_log["avg_train_cos"] = avg_train_cos
            step_log["avg_val_loss"] = avg_val_loss
            step_log["avg_val_cos"] = avg_val_cos

            # Print update to console without breaking progress bar
            train_pbar.write(f"Step {global_step} | Avg Train Loss: {avg_train_loss:.4f} | Avg Train Cosine: {avg_train_cos:.4f}")
            train_pbar.write(f"Step {global_step} | Avg Val Loss: {avg_val_loss:.4f} | Avg Val Cosine: {avg_val_cos:.4f}")

        history.append(step_log)

        # 1. Periodic Checkpoint (Every SAVE_EVERY steps)
        if global_step % SAVE_EVERY == 0:
            checkpoint_path = f"{DRIVE_SAVE_PATH}/distillation_epoch_{epoch+1}_step_{global_step}.pt"
            print(f"\n>> Step {global_step} complete. Checkpoint saved to: {checkpoint_path}")
            torch.save({
                'step': global_step,
                'epoch': epoch + 1,
                'student_state_dict': model.student.state_dict(),
                'projection_state_dict': model.projection.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': loss.item(),
            }, checkpoint_path)
            df_history = pd.DataFrame(history)
            df_history.to_csv(f"{DRIVE_SAVE_PATH}/training_log_epoch_{epoch+1}_step_{global_step}.csv", index=False)

    # --- 2. End of Epoch Checkpoint ---
    epoch_save_path = f"{DRIVE_SAVE_PATH}/distillation_epoch_{epoch+1}_step_{global_step}.pt"
    torch.save({
        'epoch': epoch + 1,
        'step': global_step,
        'student_state_dict': model.student.state_dict(),
        'projection_state_dict': model.projection.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss.item(),
    }, epoch_save_path)

    # Save training history CSV at end of epoch
    df_history = pd.DataFrame(history)
    df_history.to_csv(f"{DRIVE_SAVE_PATH}/training_log_epoch_{epoch+1}_step_{global_step}.csv", index=False)

    print(f"\n>> Epoch {epoch+1} complete. Checkpoint saved to: {epoch_save_path}")